In [ ]:
from pyspark.sql.functions import col

StatementMeta(, 9180ba70-8a22-44f1-bc8c-b56d400e6d76, 4, Finished, Available, Finished, False)

In [10]:
# Read customer table
df_gold_customers = spark.read.format("delta").load(
    "abfss://my_workspace@onelake.dfs.fabric.microsoft.com/Ecommerce_Data.Lakehouse/Files/gold/fact_customer_metrics"
)

# Filter to ONLY current records (is_current = True)
df_current = df_gold_customers.filter(col("is_current") == True)

print(f" Loaded {df_current.count():,} current customer records")
display(df_current.limit(5))

StatementMeta(, ee35e0bf-f3bb-499a-b560-28a65932ed0a, 12, Finished, Available, Finished, False)

 Loaded 7,621 current customer records


SynapseWidget(Synapse.DataFrame, 6aab2e17-3d70-439b-b970-1b798e3d6dc1)

In [12]:
# Select features
df_features = df_current.select(
    "customer_id",
    "total_orders",
    "total_spent",
    "avg_order_value",
    "age",
    "avg_rating",
    "return_count"
)

print(f"Ready for K-means with 7,621 customers")
#display(df_features.limit(5))

StatementMeta(, ee35e0bf-f3bb-499a-b560-28a65932ed0a, 14, Finished, Available, Finished, False)

Ready for K-means with 7,621 customers


In [13]:
from pyspark.ml.feature import VectorAssembler

# columns are features
feature_columns = [
    "total_orders",
    "total_spent",
    "avg_order_value",
    "age",
    "avg_rating",
    "return_count"
]

# Combine all features into one column into features
assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features"
)

df_assembled = assembler.transform(df_features)

print(f" Created feature vectors")
#display(df_assembled.select("customer_id", "features"))

StatementMeta(, ee35e0bf-f3bb-499a-b560-28a65932ed0a, 15, Finished, Available, Finished, False)

 Created feature vectors


SynapseWidget(Synapse.DataFrame, 56b22eca-f3ee-4c41-b36b-025bf4f03f45)

In [16]:
from pyspark.ml.feature import StandardScaler

# Scale all features to same range (mean=0, std=1)
scaler = StandardScaler(
    inputCol="features",
    outputCol="scaled_features",
    withMean=True,
    withStd=True
)

# Fit and transform
scaler_model = scaler.fit(df_assembled)
df_scaled = scaler_model.transform(df_assembled)

print(f"Features scaled")
display(df_scaled.select("customer_id", "features", "scaled_features").limit(3))

StatementMeta(, ee35e0bf-f3bb-499a-b560-28a65932ed0a, 18, Finished, Available, Finished, False)

Features scaled


SynapseWidget(Synapse.DataFrame, fb5e101a-0d93-409a-bb2e-7c2389a8808a)

In [17]:
from pyspark.ml.clustering import KMeans

# Create K-means model with 4 clusters
kmeans = KMeans(
    k=4,
    seed=42,
    featuresCol="scaled_features",
    predictionCol="cluster"
)

# Run clustering
kmeans_model = kmeans.fit(df_scaled)

# Add cluster assignments to data
df_clustered = kmeans_model.transform(df_scaled)

print(f"K-means clustering complete!")
print(f" Found 4 customer groups")
#display(df_clustered.select("customer_id", "cluster").limit(10))

StatementMeta(, ee35e0bf-f3bb-499a-b560-28a65932ed0a, 19, Finished, Available, Finished, False)

K-means clustering complete!
 Found 4 customer groups


In [20]:
from pyspark.sql.functions import when

# segment names
df_segmented = df_clustered.withColumn(
    "segment",
    when(col("cluster") == 0, "Frequent Customers").
    when(col("cluster") == 1, "Loyal Customers").
    when(col("cluster") == 2, "At Risk").
    when(col("cluster") == 3, "New Customers")
)

StatementMeta(, ee35e0bf-f3bb-499a-b560-28a65932ed0a, 22, Finished, Available, Finished, False)

In [21]:
# Create segmentation table with available columns
customer_segmentation = df_segmented.select(
    "customer_id",
    "total_orders",
    "total_spent",
    "avg_order_value",
    "age",
    "avg_rating",
    "return_count",
    "cluster",
    "segment"
)

print(f"✅ Created {customer_segmentation.count():,} customer segments")
display(customer_segmentation.limit(20))

StatementMeta(, ee35e0bf-f3bb-499a-b560-28a65932ed0a, 23, Finished, Available, Finished, False)

✅ Created 7,621 customer segments


SynapseWidget(Synapse.DataFrame, 2f1f32af-c898-4e4e-a2ec-8cfe273aa3de)